# Challenge 00 - Ready, Set, Connect

**Authoring draft: not tested.** This notebook can create billable resources when you explicitly opt in. Complete account, permissions, quota, and native Context Engine entitlement checks before the event. Do not run all cells blindly.

Use the published Challenge 00 guide and this notebook for setup. Credentials stay in a local `.env` file or Foundry project connections, never in cells.

## Prepare configuration

Copy `.env.sample` to `.env` using the file explorer. Choose a unique `WORKSHOP_ID`, enter your Elastic URLs and setup key, and sign in with Azure CLI in the terminal (`az login`, then `az account set --subscription ...`). Select Python (Context Quest).

Attach an existing approved Foundry project/model by entering its endpoint and deployment name. Otherwise use the opt-in provisioning cell below. The devcontainer installs packages; do not add extra model-provider accounts.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

resource_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(resource_root))
from workshop import configure, required, workshop_id, ROOT, seed_data, kibana_request, discover_tools, foundry_clients
configure()
print('Workshop:', workshop_id())
subprocess.run(['az', 'account', 'show', '--query', '{name:name,state:state}', '-o', 'table'], check=True)

## Optional: create new Azure resources

Skip this cell when attaching existing resources. Review `infra/README.md` and `main.bicep` first. Set the approved location, model/version, and Foundry project role GUID locally in the terminal environment or `.env`; these additional provisioning values are not secrets. The script creates only a dedicated, tagged resource group, a Foundry account/project, a model, and role assignments.

Deployment and role-assignment permissions are required. The template is a draft, not a tested region/model recommendation.

In [ ]:
CREATE_NEW_AZURE = False
if CREATE_NEW_AZURE:
    confirmation = input('Create billable workshop resources? Type the WORKSHOP_ID: ')
    subprocess.run(
        ['bash', str(ROOT / 'infra/deploy.sh')],
        env={**os.environ, 'CONFIRM_CREATE': confirmation}, check=True,
    )
    print('Copy the endpoint/deployment outputs into .env and rerun configuration.')
else:
    print('Provisioning skipped. Attach an existing project or deliberately opt in above.')

## Load the clean dataset

This writes only named workshop-owned indices and stable document IDs. Re-running restores bundled records; it does not delete extra records. Expected counts are 3 accounts, 4 tickets, 3 products, and 4 policies. Stop if counts differ and explain why before comparing agents. The safety fixture is not loaded yet.

In [ ]:
seed_results = seed_data()
display(seed_results)

## Confirm native Context Engine access

In the intended Kibana space, enable Context Engine in Advanced Settings. Confirm that the Context UI opens and workflow/inference access is available. This read-only API call can return an empty list before Challenge 02; a successful request is not proof that KI generation or retrieval works. A 403 suggests missing privileges; a 404 can mean the feature is disabled or unavailable.

Native feature access is required. Do not replace it with an ordinary index and label that success.

In [ ]:
native_indices = kibana_request('GET', '/api/context_engine/ai_index')
display(native_indices)

## Configure the Foundry MCP connections

Use the key-based connection guidance in the resource README. Create two Foundry remote MCP connections targeting your Elastic hosted endpoint, with header `Authorization` and value `ApiKey <runtime-key>`. Use separate baseline and Optimus keys. Save the connection names/IDs in `.env`.

The baseline key may read only workshop sources, not AI Index data. Optimus additionally needs Context Engine read privileges and its backing data. Neither key may write. Make the matching local runtime keys available for tool discovery below. Use the actual wire names to fill the three `MCP_*_TOOL` entries in `.env`. Do not grant every discovered tool.

In [ ]:
configure()
baseline_tools = await discover_tools('baseline')
display([{'name': tool['name'], 'description': tool.get('description')} for tool in baseline_tools])
# Inspect a chosen tool's inputSchema locally if needed; do not print credentials.

## Confirm project/model access and record readiness

This small model request incurs inference usage and verifies the deployment, not the MCP route or evaluator. Challenge 01 proves the Foundry-to-Elastic route. Foundry safety-evaluation access must be checked separately by the organizer before delivery.

In [ ]:
project, client = foundry_clients()
probe = client.responses.create(
    model=required('AZURE_AI_MODEL_DEPLOYMENT_NAME'),
    input='Reply with the single word Ready.',
)
print(probe.output_text)
readiness = {
    'source_counts_reviewed': False,
    'native_context_ui_and_generation_access_confirmed': False,
    'runtime_permissions_reviewed': False,
    'foundry_model_request_succeeded': probe.status == 'completed',
    'foundry_to_elastic_mcp_verified': False,
    'hosted_evaluator_verified_by_organizer': False,
}
display(readiness)

## Explain

Update only the checks you have evidence for. Which credentials can create resources, and which can only retrieve data? Which requests were local, and which came from Foundry? What will continue to incur charges after closing this browser tab?

Keep a note of blockers. Do not convert an unverified item into a successful readiness check.